# English → Khmer NMT: comparing three deep-learning approaches (Colab runner)

| | Approach | Training strategy | Trainable params |
|---|---|---|---|
| A | Transformer **from scratch** | random init, own SentencePiece vocab | ~11.5M |
| B | NLLB-200-600M **frozen backbone** | train decoder cross-attention only | ~50M (8%) |
| C | NLLB-200-600M **full fine-tuning** | train every parameter | ~615M |
| – | Zero-shot NLLB-200-600M | *reference only, not trained* | 0 |

All approaches use the same data split, and all are scored on the **same 5,000 test sentences** with the same decoding (beam 4) and metrics (chrF++, chrF, BLEU).
**Everything is stored in Google Drive** under `MyDrive/english-khmer-translation/`: data, checkpoints, models, results and figures.

**Setup (once):** upload `english-khmer-translation.zip` to **My Drive**, open this notebook in Colab, choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**If the session disconnects:** reconnect and choose **Run all** again. Finished steps are skipped, and training resumes from the latest checkpoint in Drive.

## 0. Settings

In [ ]:
# Hyperparameter tuning (grid: learning rate x dropout, scored on the VALIDATION set).
# The rubric requires tuning at least the best approach; A and C are tuned by default.
TUNE_SCRATCH = True
TUNE_FROZEN = False
TUNE_FULL = True

# Extra command-line options per approach, e.g. "--epochs 10" or "--max-steps 1000".
# Empty = the defaults in each script (A: 15 epochs with early stopping,
# B: 4,000 steps, C: 2,000 steps; effective batch 128 / 16 / 16 sentence pairs).
SCRATCH_ARGS = ""
FROZEN_ARGS = ""
FULL_ARGS = ""

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive and unzip the project

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/english-khmer-translation"

# Look for the zip anywhere in My Drive (up to 3 folders deep), including renamed
# copies such as "english-khmer-translation (1).zip". The newest one is used.
found = subprocess.run(
    ["find", DRIVE_ROOT, "-maxdepth", "3", "-iname", "english-khmer-translation*.zip"],
    capture_output=True, text=True,
).stdout.splitlines()
zips = sorted(found, key=os.path.getmtime, reverse=True)

if zips:
    ZIP_PATH = zips[0]
    print(f"Using {ZIP_PATH}")
    # Check every file's checksum first, so a damaged upload stops here instead of
    # silently leaving the previous version of the code in place.
    if subprocess.run(["unzip", "-tq", ZIP_PATH]).returncode != 0:
        raise RuntimeError(f"{ZIP_PATH} is damaged. Upload it again, then re-run this cell.")
    # The zip holds code only, so data/, checkpoints/, models/ and results/ are left untouched.
    !unzip -o -q "{ZIP_PATH}" -d "{DRIVE_ROOT}"
    print(f"Code extracted to {PROJECT_DIR}")
elif os.path.isdir(f"{PROJECT_DIR}/src"):
    print(f"No zip found; using the existing code in {PROJECT_DIR}")
else:
    print("Zip not found. This is what is in My Drive:")
    !ls -la "{DRIVE_ROOT}"
    raise FileNotFoundError("Upload english-khmer-translation.zip to My Drive, then run this cell again.")

In [ ]:
%cd /content/drive/MyDrive/english-khmer-translation

### Verify the code
Checks that every code file in Drive is byte-for-byte the version this notebook was built for, using SHA-256 fingerprints. If a file is missing or changed, the notebook stops here instead of training with the wrong code.

In [ ]:
import hashlib

VERIFY_CODE = True  # Set to False only if you changed the code on purpose.

EXPECTED_SHA256 = {
    "requirements.txt": "95d2623eca5fe2b1701ba5bc9c834ad025e1b7c3186fb4a34e04b8d440f65f45",
    "src/__init__.py": "97dccf83164d397e8db471de62e6f0f64e6ac4d77d832ec7cc237184ce7b3421",
    "src/compare.py": "08f591182e64195f82136d9479e21d13a29ed632f0f03f790d34a07dc6c51ee3",
    "src/compare_experiments.py": "9282abe13199b1f2f3de756a4a2d09d71568db4057afabe23a94c8b946eddc7a",
    "src/config.py": "b0cb45232c665aeae68f96e94aa9f686e61b10852415d2d184afe8292f806164",
    "src/data.py": "513fa4a65d42bab0f42588d225a65cefb8728d752d98fcffbd6f868c73e27be8",
    "src/data_stats.py": "f8b313497a7740762f3618509136ee31a8327bcb10d7ae64a1318705d100e333",
    "src/error_analysis.py": "7137f692ed146d7ff522f1c66dbfee7f8448962671e557b0ee19b5041c033616",
    "src/evaluate.py": "f773123651c7c42eb7c4cbf802b4ff4b3dea71af2b2dba734da1d8abf097f3bc",
    "src/model_scratch.py": "bd2408cca0311e37117c1a9a379c4ba56b82b2706e9a92ce679a5f76336bec7d",
    "src/plots.py": "ec35f107c8958d8eb3c5a5df3b75f8d38c1f7f773bdd698409d0dd08100f3a9b",
    "src/probe.py": "5cab2934d2618d353c8979be8ee53c296ce06f033972e3bb0afd08d926358c6e",
    "src/train_finetune.py": "411c2df87a863146cb9e388248e0f5ded19ac2449e937cd5d59e12f32184d51b",
    "src/train_frozen.py": "1b59505a6918da3ebbe96b9bcabef349c0f0441903ac99ca3dac6d390b9a08e2",
    "src/train_scratch.py": "1e090e7a117ee84a941fe3483e3e2c6d4ced36253b6db279ef66cbcccd378c44",
    "src/translate.py": "41fb86eca1590179fe6550119d1f2984e9244f2b0d7dbb7dc5df77118ba70a38",
    "src/tune.py": "faf5bf1e819cc21875bb6fa4592b4be25a40cbf674ddc9d1369fd2ba434ec41a",
    "src/utils/__init__.py": "2260db3bdcd3b4ebd0ffa725c1769cf83a7dfcf33b969d0119002d366ed67dca",
    "src/utils/metrics.py": "782a90e9f4991f6c7ce955e112a28c1b8592dfb3d0a0b27c9ea91ed204259682",
    "src/utils/plotting.py": "d6ffff0bb85b7056d8ac5f10f36d0ac54559a2c3d1d1ad803b983523db5634a8",
    "src/utils/reporting.py": "55e3bb437798e65b2c012ea2e2d31b74a32eebb82739061902e4a30ea57eb750",
    "src/utils/seed.py": "c8f3dd3d3909e9116b415e6214885603ed9fd9cf20b3ab7aebfa6d8b39bacef1",
    "src/utils/training.py": "5249ee051e69a2dc3c46d156b09d7d77c69a15aea5b3a673bdc7526d6e23bacd",
}

if VERIFY_CODE:
    problems = []
    for path, expected in EXPECTED_SHA256.items():
        if not os.path.isfile(path):
            problems.append(f"missing: {path}")
        elif hashlib.sha256(open(path, "rb").read()).hexdigest() != expected:
            problems.append(f"changed: {path}")
    if problems:
        raise RuntimeError("Code check failed:\n" + "\n".join(problems))
    print(f"All {len(EXPECTED_SHA256)} code files verified.")

## 3. Install requirements
Colab's preinstalled CUDA build of PyTorch is kept, and everything else is installed at the pinned versions.

In [ ]:
!grep -v "^torch==" requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 4. Helpers

In [ ]:
import glob
import json

def done(path: str) -> bool:
    """True if a step's output file exists (the step is then skipped)."""
    if os.path.exists(path):
        print(f"✓ {path} exists; skipping.")
        return True
    return False

def resume_flag(pattern: str) -> str:
    """'--resume' if a checkpoint matching pattern exists in Drive."""
    if glob.glob(pattern):
        print(f"Resuming from checkpoint ({pattern}).")
        return "--resume"
    return ""

def tuned_args(approach: str) -> str:
    """Best learning rate / dropout / weight decay found by src.tune, if any."""
    path = f"results/tuning/{approach}_best.json"
    if not os.path.isfile(path):
        return ""
    best = json.load(open(path, encoding="utf-8"))
    print(f"Using tuned hyperparameters from {path}: lr={best['learning_rate']}, "
          f"dropout={best['dropout']}, weight_decay={best['weight_decay']}")
    return (f"--learning-rate {best['learning_rate']} --dropout {best['dropout']} "
            f"--weight-decay {best['weight_decay']}")

## 5. Data: prepare the splits and describe the dataset

In [ ]:
if not done("data/processed/nllb_eng_khm"):
    !python -m src.data --dynamic-padding
if not done("results/data_stats.json"):
    !python -m src.data_stats

## 6. Reference: zero-shot NLLB (not trained, so it does not count as an approach)

In [ ]:
# Translations saved by an earlier version of this notebook are re-scored instead of
# re-translated (src.evaluate first checks they match this exact test set).
OLD_PREDICTIONS = "results/baseline_results_predictions.jsonl"

if not done("results/predictions/baseline.jsonl"):
    if os.path.isfile(OLD_PREDICTIONS):
        !python -m src.evaluate --approach baseline --from-predictions {OLD_PREDICTIONS}
    else:
        !python -m src.evaluate --approach baseline

## 7. Approach A: Transformer from scratch
Tuning grid: learning rate {3e-4, 5e-4, 1e-3} × dropout {0.1, 0.3}, 3 epochs on an 80k-pair subset per trial. The final run then uses the best configuration, with early stopping on validation chrF++.

In [ ]:
if TUNE_SCRATCH and not done("results/tuning/scratch_best.json"):
    !python -m src.tune --approach scratch

if not done("results/scratch_train_summary.json"):
    args = f"{tuned_args('scratch')} {SCRATCH_ARGS} {resume_flag('checkpoints/scratch/last.pt')}"
    !python -m src.train_scratch {args}

if not done("results/scratch_results.json"):
    !python -m src.evaluate --approach scratch

## 8. Approach B: NLLB with a frozen backbone
Only the decoder cross-attention layers are trained (~8% of parameters).

In [ ]:
if TUNE_FROZEN and not done("results/tuning/frozen_best.json"):
    !python -m src.tune --approach frozen

if not done("results/frozen_train_summary.json"):
    args = f"{tuned_args('frozen')} {FROZEN_ARGS} {resume_flag('checkpoints/frozen/checkpoint-*')}"
    !python -m src.train_frozen {args}

if not done("results/frozen_results.json"):
    !python -m src.evaluate --approach frozen

## 9. Approach C: NLLB full fine-tuning
Every parameter is trained. Adafactor, gradient checkpointing and fp16 keep memory low.
Tuning grid: learning rate {1e-5, 3e-5, 1e-4} × dropout {0.1, 0.3}, 300 steps per trial.

In [ ]:
if TUNE_FULL and not done("results/tuning/full_best.json"):
    !python -m src.tune --approach full

if not done("results/full_train_summary.json"):
    args = f"{tuned_args('full')} {FULL_ARGS} {resume_flag('checkpoints/full_finetune/checkpoint-*')}"
    !python -m src.train_finetune {args}

if not done("results/full_results.json"):
    !python -m src.evaluate --approach full

### Free up Drive space (optional)
Checkpoints are only needed while training. Once an approach's `results/<approach>_train_summary.json` exists, its checkpoints can go.
Files deleted from a mounted Drive go to the Drive trash, and still count against your quota until the trash is emptied.

In [ ]:
# !rm -rf checkpoints/frozen checkpoints/full_finetune checkpoints/scratch

## 10. Analysis: error analysis, comparison table and figures

In [ ]:
!python -m src.error_analysis
!python -m src.compare
!python -m src.plots

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("results/comparison.md", encoding="utf-8").read()))
for name in ["test_metrics", "learning_curves", "val_chrf_curves", "efficiency",
             "tuning_scratch", "tuning_full", "error_categories", "chrf_by_length",
             "length_distribution"]:
    path = f"results/figures/{name}.png"
    if os.path.isfile(path):
        display(Markdown(f"**{name}**"), Image(path, width=800))

## 11. Download the results for GitHub
Creates `results_bundle.zip` in Drive with `results/` (metrics, logs, predictions, tuning, error analysis, figures) and the small scratch model. Unzip it into your local repository and commit it.
The NLLB models (~1.2 GB each) stay in Drive under `models/`. Share that folder and put the link in the README.

In [ ]:
!rm -f results_bundle.zip
!zip -q -r results_bundle.zip results models/scratch -x "models/scratch/spm.vocab"
!ls -lh results_bundle.zip